**Machine Learning Project 4**

In [1]:
from google.colab import files
import pandas as pd

# TASK 1. Downloading data
print("TASK 1. Downloading data")
print("Status: Uploading training.csv")
uploaded = files.upload()
train_filename = list(uploaded.keys())[0]
train_df = pd.read_csv(train_filename)

print("\nStatus: Uploading test.csv")
uploaded = files.upload()
test_filename = list(uploaded.keys())[0]
test_df = pd.read_csv(test_filename)

print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print("\nStatus: Data loaded successfully.")

TASK 1. Downloading data
Status: Uploading training.csv


Saving training.csv to training.csv

Status: Uploading test.csv


Saving test.csv to test.csv

Train shape: (72983, 34)
Test shape: (48707, 33)

Status: Data loaded successfully.


In [2]:
!pip install scikit-learn pandas numpy matplotlib seaborn -q

In [3]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [4]:
# TASK 2. Splitting data

print("TASK 2. Splitting data")
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# List of columns
print("\nList of columns:")
print(train_df.columns)

print("\nTrain set head (checking data examples):")
print(train_df.head())

print("- "*40)
print("\nTarget: 'IsBadBuy'")
print(f"\nTarget distribution:\n{train_df['IsBadBuy'].value_counts(normalize=True)}")

# Converting PurchDate to datetime
train_df['PurchDate'] = pd.to_datetime(train_df['PurchDate'])

# Sorting by date
train_df = train_df.sort_values('PurchDate').reset_index(drop=True)

# Creating time-based split (1/3 each for train, validation, test)
print("- "*40)
print("\nStatus: splitting dataset. 1/3 each for train, validation, test")
# Comment: finding boundary dates, assigning all records on boundary date to earlier set
n = len(train_df)
train_idx = int(n * 1/3)
valid_idx = int(n * 2/3)

# Getting boundary dates (dates at the split points)
train_boundary_date = train_df.iloc[train_idx]['PurchDate']
valid_boundary_date = train_df.iloc[valid_idx]['PurchDate']

# Creating splits
df_train = train_df[train_df['PurchDate'] < train_boundary_date].copy()
df_valid = train_df[(train_df['PurchDate'] >= train_boundary_date) &
                    (train_df['PurchDate'] < valid_boundary_date)].copy()
df_test = train_df[train_df['PurchDate'] >= valid_boundary_date].copy()

print(f"Train dates: {df_train['PurchDate'].min()} to {df_train['PurchDate'].max()}")
print(f"Valid dates: {df_valid['PurchDate'].min()} to {df_valid['PurchDate'].max()}")
print(f"Test dates: {df_test['PurchDate'].min()} to {df_test['PurchDate'].max()}")
print(f"\nSplit sizes. Train: {len(df_train)}. Valid: {len(df_valid)}. Test: {len(df_test)}")
print(f"Total: {len(df_train) + len(df_valid) + len(df_test)} (original: {len(train_df)})")
print(f"\nVerifying no overlap:")
print(f"Train max < Valid min: {df_train['PurchDate'].max() < df_valid['PurchDate'].min()}")
print(f"Valid max < Test min: {df_valid['PurchDate'].max() < df_test['PurchDate'].min()}")


TASK 2. Splitting data

Train shape: (72983, 34)
Test shape: (48707, 33)

List of columns:
Index(['RefId', 'IsBadBuy', 'PurchDate', 'Auction', 'VehYear', 'VehicleAge',
       'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
       'WheelTypeID', 'WheelType', 'VehOdo', 'Nationality', 'Size',
       'TopThreeAmericanName', 'MMRAcquisitionAuctionAveragePrice',
       'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice',
       'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice',
       'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice',
       'MMRCurrentRetailCleanPrice', 'PRIMEUNIT', 'AUCGUART', 'BYRNO',
       'VNZIP1', 'VNST', 'VehBCost', 'IsOnlineSale', 'WarrantyCost'],
      dtype='object')

Train set head (checking data examples):
   RefId  IsBadBuy  PurchDate Auction  VehYear  VehicleAge   Make  \
0      1         0  12/7/2009   ADESA     2006           3  MAZDA   
1      2         0  12/7/2009   ADESA     2004           5  DOD

In [5]:
# TASK 3. Preprocessing categorical variables
print("TASK 3. Preprocessing categorical variables")

def preprocess_data(train, valid, test, use_count_encoding=True):

    # Targets for each data set
    y_train = train['IsBadBuy'].values
    y_valid = valid['IsBadBuy'].values
    y_test = test['IsBadBuy'].values

    # Dropping target and identifiers
    drop_cols = ['IsBadBuy', 'RefId', 'PurchDate']
    X_train = train.drop(columns=drop_cols)
    X_valid = valid.drop(columns=drop_cols)
    X_test = test.drop(columns=drop_cols)

    # Identifying categorical and numerical columns
    categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
    numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    print(f"\nCategorical columns: {len(categorical_cols)}")
    print(f"Numerical columns: {len(numerical_cols)}")

    # Handling missing values in numerical columns
    for col in numerical_cols:
        mean_val = X_train[col].mean()
        X_train[col].fillna(mean_val, inplace=True)
        X_valid[col].fillna(mean_val, inplace=True)
        X_test[col].fillna(mean_val, inplace=True)

    # Handling categorical variables
    if use_count_encoding:
        for col in categorical_cols:
            counts = X_train[col].value_counts().to_dict() # counts on training data
            ## Applying to all data sets, using 0 for unseen categories:
            X_train[col] = X_train[col].map(counts).fillna(0)
            X_valid[col] = X_valid[col].map(counts).fillna(0)
            X_test[col] = X_test[col].map(counts).fillna(0)
    else:
        encoders = {}
        for col in categorical_cols:
            le = LabelEncoder() ## label encoding for new categories
            # Fitting on training data
            X_train[col] = X_train[col].fillna('MISSING')
            le.fit(X_train[col])
            encoders[col] = le

            ## Transforming training
            X_train[col] = le.transform(X_train[col])

            # Transforming validation/test with unseen category handling
            X_valid[col] = X_valid[col].fillna('MISSING')
            X_valid[col] = X_valid[col].apply(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )

            X_test[col] = X_test[col].fillna('MISSING')
            X_test[col] = X_test[col].apply(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )

    return X_train, X_valid, X_test, y_train, y_valid, y_test


X_train, X_valid, X_test, y_train, y_valid, y_test = preprocess_data(
    df_train, df_valid, df_test, use_count_encoding=True
)

print(f"\nProcessed shapes")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape} | y_valid: {y_valid.shape}")
print(f"X_test: {X_test.shape}  | y_test: {y_test.shape}")

TASK 3. Preprocessing categorical variables

Categorical columns: 14
Numerical columns: 17

Processed shapes
X_train: (24232, 31) | y_train: (24232,)
X_valid: (24414, 31) | y_valid: (24414,)
X_test: (24337, 31)  | y_test: (24337,)


In [6]:
# TASK 4. Train LogisticRegression, GaussianNB, KNN from sklearn
print("TASK 4. Train LogisticRegression, GaussianNB, KNN from sklearn")

# Normalizing the data before training the models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

# Training models
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'GaussianNB': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}
for name, model in models.items():
    print(f"\nStatus: Training {name}.")
    model.fit(X_train_scaled, y_train)

    # Predicting probabilities
    y_pred_proba = model.predict_proba(X_valid_scaled)[:, 1]

    # Calculating metrics
    auc = roc_auc_score(y_valid, y_pred_proba)
    gini = 2 * auc - 1

    results[name] = {'model': model, 'auc': auc, 'gini': gini}
    print(f"{name} - Validation AUC: {auc:.4f}, Gini: {gini:.4f}")

# Checking if the 0.15 Gini requirement is met
print("\nChecking whether the 0.15 Gini requirement is met")
requirement_met = False

for name, metrics in results.items():
    gini = metrics['gini']
    print(f"{name:20s} - Gini: {gini:.4f} - {'PASS' if gini >= 0.15 else 'FAIL'}")

    if gini >= 0.15:
        requirement_met = True

if requirement_met:
    print("Requirement met: at least one model achieved Gini >= 0.15")
else:
    print("REQUIREMENT NOT MET: no model achieved Gini >= 0.15")


# Finding the best model
best_model_name = max(results, key=lambda x: results[x]['gini'])
print(f"\nBest model: {best_model_name} with Gini: {results[best_model_name]['gini']:.4f}")

print()
print(results)


TASK 4. Train LogisticRegression, GaussianNB, KNN from sklearn

Status: Training LogisticRegression.
LogisticRegression - Validation AUC: 0.7325, Gini: 0.4651

Status: Training GaussianNB.
GaussianNB - Validation AUC: 0.7246, Gini: 0.4491

Status: Training KNN.
KNN - Validation AUC: 0.6614, Gini: 0.3228

Checking whether the 0.15 Gini requirement is met
LogisticRegression   - Gini: 0.4651 - PASS
GaussianNB           - Gini: 0.4491 - PASS
KNN                  - Gini: 0.3228 - PASS
Requirement met: at least one model achieved Gini >= 0.15

Best model: LogisticRegression with Gini: 0.4651

{'LogisticRegression': {'model': LogisticRegression(max_iter=1000, random_state=42), 'auc': np.float64(0.7325330308521331), 'gini': np.float64(0.4650660617042661)}, 'GaussianNB': {'model': GaussianNB(), 'auc': np.float64(0.7245625281091543), 'gini': np.float64(0.44912505621830867)}, 'KNN': {'model': KNeighborsClassifier(), 'auc': np.float64(0.6614027788161106), 'gini': np.float64(0.3228055576322213)}}


In [7]:
# TASK 5. Implementing Gini score calculation
print("TASK 5. Implementing Gini score calculation")
def calculate_roc_auc(y_true, y_pred_proba):

    # Creating pairs of (probability, true_label)
    pairs = list(zip(y_pred_proba, y_true))
    # Sorting by predicted probability (descending order)
    pairs.sort(reverse=True, key=lambda x: x[0])

    n_pos = sum(y_true)
    n_neg = len(y_true) - n_pos

    if n_pos == 0 or n_neg == 0:
        return 0.5

    # Counting concordant pairs
    sum_ranks = 0
    pos_count = 0

    for i, (prob, label) in enumerate(pairs):
        if label == 1:
            sum_ranks += (len(pairs) - i - 1)
            pos_count += 1

    #Subtracting pairs where both are positive (shouldn't be counted)
    sum_ranks -= (pos_count * (pos_count - 1)) / 2

    auc = sum_ranks / (n_pos * n_neg)
    return auc


def calculate_gini(y_true, y_pred_proba):
    auc = calculate_roc_auc(y_true, y_pred_proba)
    return 2 * auc - 1


print("Validating Custom Gini Implementation")
# Testing on LogisticRegression predictions
lr_model = results['LogisticRegression']['model']
y_pred_proba = lr_model.predict_proba(X_valid_scaled)[:, 1]

custom_auc = calculate_roc_auc(y_valid, y_pred_proba)
sklearn_auc = roc_auc_score(y_valid, y_pred_proba)
custom_gini = calculate_gini(y_valid, y_pred_proba)
sklearn_gini = 2 * sklearn_auc - 1

print(f"\nCustom AUC: {custom_auc:.6f}")
print(f"Sklearn AUC: {sklearn_auc:.6f}")
print(f"Difference: {abs(custom_auc - sklearn_auc):.6f}")
print(f"\nCustom Gini: {custom_gini:.6f}")
print(f"Sklearn Gini: {sklearn_gini:.6f}")
print(f"Difference: {abs(custom_gini - sklearn_gini):.6f}")


TASK 5. Implementing Gini score calculation
Validating Custom Gini Implementation

Custom AUC: 0.732533
Sklearn AUC: 0.732533
Difference: 0.000000

Custom Gini: 0.465066
Sklearn Gini: 0.465066
Difference: 0.000000


In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# TASK 6. Custom model implementations
print("TASK 6. Custom model implementations")

class CustomLogisticRegression:

    def __init__(self, learning_rate=0.01, n_iterations=1000, batch_size=32,
                 regularization=None, lambda_reg=0.01):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.batch_size = batch_size
        self.regularization = regularization
        self.lambda_reg = lambda_reg
        self.w = None
        self.b = None

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500) # clip to prevent overflow
        positive = z >= 0
        result = np.zeros_like(z, dtype=float)

        result[positive] = 1 / (1 + np.exp(-z[positive])) ## for positive values: 1 / (1 + exp(-z))

        exp_z = np.exp(z[~positive])
        result[~positive] = exp_z / (1 + exp_z) ## for negative values: exp(z) / (1 + exp(z))

        return result

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Initializing parameters
        self.w = np.zeros(n_features)
        self.coef_ = self.w
        self.b = 0
        self.intercept_ = self.b

        # SGD
        for iteration in range(self.n_iter):
            # Shuffling data
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            # Mini-batch gradient descent
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i+self.batch_size]
                y_batch = y_shuffled[i:i+self.batch_size]

                # Forward pass
                linear_model = np.dot(X_batch, self.w) + self.b
                y_pred = self._sigmoid(linear_model)

                # Computing gradients
                batch_size = len(y_batch)
                dw = (1/batch_size) * np.dot(X_batch.T, (y_pred - y_batch))
                db = (1/batch_size) * np.sum(y_pred - y_batch)

                # Adding regularization
                if self.regularization == 'l1':
                    dw += self.lambda_reg * np.sign(self.w)
                elif self.regularization == 'l2':
                    dw += self.lambda_reg * self.w

                # Updating parameters
                self.w -= self.lr * dw
                self.b -= self.lr * db

        return self

    def predict_proba(self, X):    # predicting probabilities
        linear_model = np.dot(X, self.w) + self.b
        proba = self._sigmoid(linear_model)
        return np.vstack([1-proba, proba]).T

    def predict(self, X, threshold=0.5):   #predicting class labels
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


class CustomKNN:

    def __init__(self, n_neighbors=5):
        self.k = n_neighbors
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        return self

    def _euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2, axis=1))

    def predict_proba(self, X):
        probas = []
        for x in X:
            # Calculating distances to all training points:
            distances = self._euclidean_distance(self.X_train, x)

            # Getting k nearest neighbors
            k_indices = np.argsort(distances)[:self.k]
            k_labels = self.y_train[k_indices]

            # Calculating probability as proportion of positive class
            proba_positive = np.sum(k_labels) / self.k
            probas.append([1 - proba_positive, proba_positive])

        return np.array(probas)

    def predict(self, X, threshold=0.5):    # predicting class labels
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


class CustomNaiveBayes:

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing
        self.classes = None
        self.class_prior = None
        self.means = None
        self.vars = None

    def fit(self, X, y):   # calculating class priors and feature statistics
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        n_features = X.shape[1]

        self.means = np.zeros((n_classes, n_features))
        self.vars = np.zeros((n_classes, n_features))
        self.class_prior = np.zeros(n_classes)

        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.means[idx, :] = X_c.mean(axis=0)
            self.vars[idx, :] = X_c.var(axis=0) + self.var_smoothing
            self.class_prior[idx] = X_c.shape[0] / X.shape[0]

        return self

    def _calculate_likelihood(self, x, mean, var):  ## calculating Gaussian likelihood
        eps = 1e-10
        var = np.maximum(var, eps)  ## ensuring variance is positive
        coef = 1.0 / np.sqrt(2.0 * np.pi * var)
        exponent = np.exp(-np.clip((x - mean) ** 2 / (2 * var), -500, 500))
        return np.maximum(coef * exponent, eps)  ## avoiding zero likelihood

    def predict_proba(self, X):
        probas = []

        for x in X:
            posteriors = []

            for idx, c in enumerate(self.classes):
                # Calculating prior
                prior = np.log(self.class_prior[idx] + 1e-10)

                # Calculating likelihood using log to avoid underflow
                likelihoods = self._calculate_likelihood(x, self.means[idx], self.vars[idx])
                likelihood = np.sum(np.log(likelihoods + 1e-10))

                posterior = prior + likelihood
                posteriors.append(posterior)

            # Converting to probabilities with numerical stability
            posteriors = np.array(posteriors)
            posteriors = posteriors - np.max(posteriors)  ## subtracting max for stability
            posteriors = np.exp(posteriors)
            posteriors = posteriors / (np.sum(posteriors) + 1e-10)
            probas.append(posteriors)

        return np.array(probas)

    def predict(self, X, threshold=0.5):   ## predicting class labels
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

###### ---------------------------------------


# Training custom models
custom_models = {
    'Custom LR': CustomLogisticRegression(learning_rate=0.01, n_iterations=100),
    'Custom KNN': CustomKNN(n_neighbors=5),
    'Custom NB': CustomNaiveBayes()
}

custom_results = {}
for name, model in custom_models.items():
    print(f"\nStatus: Training {name}.")
    try:
        model.fit(X_train_scaled, y_train)

        # Predict
        y_pred_proba = model.predict_proba(X_valid_scaled)[:, 1]

        # Checking for NaN
        if np.any(np.isnan(y_pred_proba)):
            print(f"  WARNING: {name} produced {np.sum(np.isnan(y_pred_proba))} NaN predictions!")
            print(f"  Min prob: {np.nanmin(y_pred_proba)}, Max prob: {np.nanmax(y_pred_proba)}")
            ## Replacing NaN with 0.5 (neutral prediction):
            y_pred_proba = np.nan_to_num(y_pred_proba, nan=0.5)

        # Calculating metrics
        auc = roc_auc_score(y_valid, y_pred_proba)
        gini = 2 * auc - 1

        custom_results[name] = {'auc': auc, 'gini': gini}
        print(f"{name} - Validation AUC: {auc:.4f}, Gini: {gini:.4f}")
    except Exception as e:
        print(f"  ERROR in {name}: {str(e)}")
        continue

# Display results
print("\nCustom model results summary")
for name, metrics in custom_results.items():
    print(f"{name:10s}  AUC: {metrics['auc']:.4f}, Gini: {metrics['gini']:.4f}")


TASK 6. Custom model implementations

Status: Training Custom LR.
Custom LR - Validation AUC: 0.7309, Gini: 0.4617

Status: Training Custom KNN.
Custom KNN - Validation AUC: 0.6614, Gini: 0.3228

Status: Training Custom NB.
Custom NB - Validation AUC: 0.7239, Gini: 0.4477

Custom model results summary
Custom LR   AUC: 0.7309, Gini: 0.4617
Custom KNN  AUC: 0.6614, Gini: 0.3228
Custom NB   AUC: 0.7239, Gini: 0.4477


In [9]:
# TASK 7. Feature engineering
print("TASK 7. Feature engineering")

def create_engineered_features(train, valid, test):

    # Making copies to avoid modifying original data
    X_train = train.copy()
    X_valid = valid.copy()
    X_test = test.copy()

    print("\nInitial shapes")
    print(f"Train: {X_train.shape}")
    print(f"Valid: {X_valid.shape}")
    print(f"Test: {X_test.shape}")

    # Identifying numerical columns (for fractions)
    numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    # Identifying categorical columns (for groupby)
    categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist() ##only object types

    print(f"Numerical columns available: {len(numerical_cols)}")
    print(f"Categorical columns available: {len(categorical_cols)}")

    # 1. CREATING FRACTION FEATURES
    potential_pairs = [
        ('VehOdo', 'VehicleAge'),
        ('MMRAcquisitionAuctionAveragePrice', 'VehBCost'),
        ('WarrantyCost', 'VehBCost'),
    ]

    print("\nStatus: Creating fraction features.")
    for col1, col2 in potential_pairs:
        if col1 in numerical_cols and col2 in numerical_cols:
            try:
                feat_name = f'{col1}_div_{col2}'

                # Processing in smaller chunks to save memory
                print(f"Creating {feat_name}.", end=" ")

                # Creating feature for each dataset
                X_train[feat_name] = X_train[col1] / (X_train[col2].abs() + 1)
                X_valid[feat_name] = X_valid[col1] / (X_valid[col2].abs() + 1)
                X_test[feat_name] = X_test[col1] / (X_test[col2].abs() + 1)

                # Handling inf values
                median_val = X_train[feat_name].replace([np.inf, -np.inf], np.nan).median()
                X_train[feat_name].replace([np.inf, -np.inf], median_val, inplace=True)
                X_valid[feat_name].replace([np.inf, -np.inf], median_val, inplace=True)
                X_test[feat_name].replace([np.inf, -np.inf], median_val, inplace=True)

                # Filling NaN
                X_train[feat_name].fillna(median_val, inplace=True)
                X_valid[feat_name].fillna(median_val, inplace=True)
                X_test[feat_name].fillna(median_val, inplace=True)

                print("Done")
            except Exception as e:
                print(f"! Error: {str(e)}")
        else:
            print(f"  Skipped {col1}/{col2} - columns not found")

    # 2. CREATING GROUPBY AGGREGATION FEATURES
    print("\nStatus: Creating groupby features.")

    groupby_configs = [
        ('Make', 'VehOdo'),
        ('Color', 'VehicleAge'),
        ('Make', 'VehBCost'),
        ('Transmission', 'VehOdo'),
    ]

    for cat_col, num_col in groupby_configs:
        if cat_col in categorical_cols and num_col in numerical_cols:
            try:
                feat_name = f'{cat_col}_mean_{num_col}'
                print(f"Creating {feat_name}.", end=" ")

                # Calculating mean on training data
                group_means = X_train.groupby(cat_col, observed=True)[num_col].mean().to_dict() # creating a dict, not a df

                # Mapping
                X_train[feat_name] = X_train[cat_col].map(group_means)
                X_valid[feat_name] = X_valid[cat_col].map(group_means)
                X_test[feat_name] = X_test[cat_col].map(group_means)

                # Filling missing with global mean
                global_mean = X_train[num_col].mean()
                X_train[feat_name].fillna(global_mean, inplace=True)
                X_valid[feat_name].fillna(global_mean, inplace=True)
                X_test[feat_name].fillna(global_mean, inplace=True)

                # Freeing memory
                del group_means

                print("Done")
            except Exception as e:
                print(f"! Error: {str(e)}")
        else:
            print(f"  Skipped {cat_col} groupby {num_col} - columns not found")

    print(f"\nFinal shapes with engineered features")
    print(f"Train: {X_train.shape}")
    print(f"Valid: {X_valid.shape}")
    print(f"Test: {X_test.shape}")

    # Verification
    for name, df in [('Train', X_train), ('Valid', X_valid), ('Test', X_test)]:
        numeric_df = df.select_dtypes(include=[np.number])
        n_inf = np.isinf(numeric_df.values).sum()
        n_nan = np.isnan(numeric_df.values).sum()
        if n_inf > 0 or n_nan > 0:
            print(f"WARNING {name}: {n_inf} inf values, {n_nan} NaN values.")

    return X_train, X_valid, X_test

###### ------------------------------------------


# Running on the dataset
df_train_eng, df_valid_eng, df_test_eng = create_engineered_features(
    df_train, df_valid, df_test
)

# Preprocessing
X_train_eng, X_valid_eng, X_test_eng, y_train, y_valid, y_test = preprocess_data(
    df_train_eng, df_valid_eng, df_test_eng
)

# Scaling
scaler = StandardScaler()
X_train_eng_scaled = scaler.fit_transform(X_train_eng)
X_valid_eng_scaled = scaler.transform(X_valid_eng)

# Training model with new features
lr_eng = LogisticRegression(max_iter=1000)
lr_eng.fit(X_train_eng_scaled, y_train)
y_pred_eng = lr_eng.predict_proba(X_valid_eng_scaled)[:, 1]

gini_eng = 2 * roc_auc_score(y_valid, y_pred_eng) - 1
print(f"\nGini with engineered features: {gini_eng:.4f}")
print(f"Improvement: {gini_eng - results['LogisticRegression']['gini']:.4f}")


TASK 7. Feature engineering

Initial shapes
Train: (24232, 34)
Valid: (24414, 34)
Test: (24337, 34)
Numerical columns available: 19
Categorical columns available: 14

Status: Creating fraction features.
Creating VehOdo_div_VehicleAge. Done
Creating MMRAcquisitionAuctionAveragePrice_div_VehBCost. Done
Creating WarrantyCost_div_VehBCost. Done

Status: Creating groupby features.
Creating Make_mean_VehOdo. Done
Creating Color_mean_VehicleAge. Done
Creating Make_mean_VehBCost. Done
Creating Transmission_mean_VehOdo. Done

Final shapes with engineered features
Train: (24232, 41)
Valid: (24414, 41)
Test: (24337, 41)
WARNING Train: 0 inf values, 1972 NaN values.
WARNING Valid: 0 inf values, 1413 NaN values.
WARNING Test: 0 inf values, 1116 NaN values.

Categorical columns: 14
Numerical columns: 24

Gini with engineered features: 0.4685
Improvement: 0.0035


In [10]:
print(custom_models['Custom LR'])

In [11]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt


# TASK 8. Feature selection
print("TASK 8. Feature selection")

def analyze_feature_importance(lr_model, feature_names, top_n=20):
    # analyzing feature importance from logistic regression coefficients
    lr_model = custom_models['Custom LR']

    if not hasattr(lr_model, 'coef_'):
        coefficients = lr_model.coef_
    elif hasattr(lr_model, 'w'):
        coefficients = lr_model.w
    else:
        print("\nModel doesn't have coefficients")
        return None

    coefs = lr_model.coef_[0]
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefs,
        'abs_coefficient': np.abs(coefs)
    }).sort_values('abs_coefficient', ascending=False)

    print("\nTop features by absolute coefficient:")
    print(feature_importance.head(top_n))

    return feature_importance


def manual_feature_selection(X_train, X_valid, y_train, y_valid,
                            feature_names, threshold=0.01):

    from sklearn.preprocessing import StandardScaler

    # Converting to numpy
    if hasattr(X_train, 'values'):
        X_train_array = X_train.values
        X_valid_array = X_valid.values
    else:
        X_train_array = X_train
        X_valid_array = X_valid

    # Training model
    lr_model = LogisticRegression(max_iter=1000, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_array)
    X_valid_scaled = scaler.transform(X_valid_array)

    lr_model.fit(X_train_scaled, y_train)

    # Getting feature importance
    coefs = np.abs(lr_model.coef_[0])

    # Selecting features above threshold
    important_features = coefs > threshold
    n_selected = np.sum(important_features)

    print(f"\nManual selection (threshold={threshold}):")
    print(f"Selected {n_selected}/{len(feature_names)} features")

    # Training on selected features
    X_train_selected = X_train_array[:, important_features]
    X_valid_selected = X_valid_array[:, important_features]

    scaler_selected = StandardScaler()
    X_train_selected_scaled = scaler_selected.fit_transform(X_train_selected)
    X_valid_selected_scaled = scaler_selected.transform(X_valid_selected)

    model_selected = LogisticRegression(max_iter=1000, random_state=42)
    model_selected.fit(X_train_selected_scaled, y_train)

    y_pred = model_selected.predict_proba(X_valid_selected_scaled)[:, 1]
    gini = 2 * roc_auc_score(y_valid, y_pred) - 1

    print(f"Gini score: {gini:.4f}")

    return important_features, gini, model_selected


def l1_feature_selection(X_train, X_valid, y_train, y_valid, C_values=[0.01, 0.1, 1, 10]):

    from sklearn.preprocessing import StandardScaler

    # Converting to numpy
    if hasattr(X_train, 'values'):
        X_train_array = X_train.values
        X_valid_array = X_valid.values
    else:
        X_train_array = X_train
        X_valid_array = X_valid

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_array)
    X_valid_scaled = scaler.transform(X_valid_array)

    results = []

    print("\nL1 Regularization Feature Selection")
    for C in C_values:
        lr_model = LogisticRegression(penalty='l1', C=C, solver='liblinear',
                                   max_iter=1000, random_state=42)
        lr_model.fit(X_train_scaled, y_train)

        # Counting non-zero coeffs
        n_features = np.sum(lr_model.coef_[0] != 0)

        # Evaluation
        y_pred = lr_model.predict_proba(X_valid_scaled)[:, 1]
        gini = 2 * roc_auc_score(y_valid, y_pred) - 1

        results.append({
            'C': C,
            'n_features': n_features,
            'gini': gini,
            'model': lr_model
        })

        print(f"C = {C:.2f}: {n_features:4d} features, Gini={gini:.4f}")

    # Finding the best
    best_result = max(results, key=lambda x: x['gini'])
    print(f"\nBest L1 model: C={best_result['C']}, "
          f"{best_result['n_features']} features, Gini={best_result['gini']:.4f}")

    return results, best_result


feature_names = X_train.columns.tolist()

# Analyzing importance
importance = analyze_feature_importance(lr_model, feature_names)

# Manual selection
important_feats, manual_gini, manual_model = manual_feature_selection(
    X_train, X_valid, y_train, y_valid, feature_names, threshold=0.01
)

# L1 selection
l1_results, best_l1 = l1_feature_selection(
    X_train, X_valid, y_train, y_valid, C_values=[0.01, 0.05, 0.1, 0.5, 1, 5, 10]
)


TASK 8. Feature selection

Top features by absolute coefficient:
                              feature  coefficient  abs_coefficient
0                             Auction    -0.007938         0.007938
1                             VehYear    -0.007938         0.007938
2                          VehicleAge    -0.007938         0.007938
3                                Make    -0.007938         0.007938
4                               Model    -0.007938         0.007938
5                                Trim    -0.007938         0.007938
6                            SubModel    -0.007938         0.007938
7                               Color    -0.007938         0.007938
8                        Transmission    -0.007938         0.007938
9                         WheelTypeID    -0.007938         0.007938
10                          WheelType    -0.007938         0.007938
11                             VehOdo    -0.007938         0.007938
12                        Nationality    -0.007938 

In [12]:
# Extractibg best model
best_model = best_l1['model']

# Getting feature names
feature_names = X_train.columns.tolist()

# Getting coeffs
coefficients = best_model.coef_[0]

# Creating feature importance DataFrame
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients),
    'selected': coefficients != 0  # True = selected, False = zero
})

# Sorting by absolute coeff
feature_importance = feature_importance.sort_values('abs_coefficient', ascending=False)

# Printing selected features
print("TOP 15 FEATURES (Selected by L1 with C=0.01)")
selected_features = feature_importance[feature_importance['selected']]
print(selected_features.to_string(index=False))

print(f"\n{len(selected_features)} features selected")

# Printing removed features
print("\n" + "-"*80)
print("Removed Features (coefficient = 0):")
removed_features = feature_importance[~feature_importance['selected']]
print(removed_features['feature'].tolist())
print(f"\n{len(removed_features)} features removed")


TOP 15 FEATURES (Selected by L1 with C=0.01)
             feature  coefficient  abs_coefficient  selected
           WheelType    -0.646811         0.646811      True
         WheelTypeID    -0.347235         0.347235      True
             VehYear    -0.291295         0.291295      True
            VehBCost    -0.214776         0.214776      True
              VehOdo     0.123596         0.123596      True
               BYRNO    -0.102995         0.102995      True
               Model    -0.078656         0.078656      True
TopThreeAmericanName    -0.041789         0.041789      True
            SubModel     0.024934         0.024934      True
            AUCGUART     0.021733         0.021733      True
              VNZIP1     0.016543         0.016543      True
        Transmission     0.012180         0.012180      True
           PRIMEUNIT     0.009645         0.009645      True
        WarrantyCost     0.007194         0.007194      True
                VNST     0.005388       

In [13]:
# TASK 9. Hyperparameters tuning
print("TASK 9. Hyperparameters tuning")

def tune_logistic_regression(X_train, X_valid, y_train, y_valid):
    from sklearn.preprocessing import StandardScaler

    # Converting to numpy
    if hasattr(X_train, 'values'):
        X_train_array = X_train.values
        X_valid_array = X_valid.values
    else:
        X_train_array = X_train
        X_valid_array = X_valid

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_array)
    X_valid_scaled = scaler.transform(X_valid_array)

    param_grid = {
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'penalty': ['l2'],
        'solver': ['lbfgs'],
        'max_iter': [1000]
    }

    print("\nStatus: Tuning Logistic Regression.")
    best_gini = 0
    best_params = None

    for C in param_grid['C']:
        model = LogisticRegression(C=C, penalty='l1', solver='saga',
                                   max_iter=1000, random_state=42)
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict_proba(X_valid_scaled)[:, 1]
        gini = 2 * roc_auc_score(y_valid, y_pred) - 1

        print(f"C = {C:.3f}: Gini={gini:.4f}")

        if gini > best_gini:
            best_gini = gini
            best_params = {'C': C}
            best_model = model

    print(f"\nBest params: {best_params}")
    print(f"Best Gini: {best_gini:.4f}")

    return best_model, best_params, best_gini


def tune_knn(X_train, X_valid, y_train, y_valid):
    from sklearn.preprocessing import StandardScaler
    from sklearn.neighbors import KNeighborsClassifier

    # Converting to numpy
    if hasattr(X_train, 'values'):
        X_train_array = X_train.values
        X_valid_array = X_valid.values
    else:
        X_train_array = X_train
        X_valid_array = X_valid

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_array)
    X_valid_scaled = scaler.transform(X_valid_array)

    k_values = [3, 5, 7, 10, 15, 20, 30, 50]

    print("\nStatus: Tuning KNN.")
    best_gini = 0
    best_k = None

    for k in k_values:
        model = KNeighborsClassifier(n_neighbors=k)
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict_proba(X_valid_scaled)[:, 1]
        gini = 2 * roc_auc_score(y_valid, y_pred) - 1

        print(f"k = {k:d}: Gini={gini:.4f}")

        if gini > best_gini:
            best_gini = gini
            best_k = k
            best_model = model

    print(f"\nBest k: {best_k}")
    print(f"Best Gini: {best_gini:.4f}")

    return best_model, best_k, best_gini


def tune_naive_bayes(X_train, X_valid, y_train, y_valid):

    from sklearn.preprocessing import StandardScaler

    # Converting to numpy
    if hasattr(X_train, 'values'):
        X_train_array = X_train.values
        X_valid_array = X_valid.values
    else:
        X_train_array = X_train
        X_valid_array = X_valid

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_array)
    X_valid_scaled = scaler.transform(X_valid_array)

    # Defining values to test
    var_smoothing_values = [1e-12, 1e-10, 1e-9, 1e-8, 1e-6]

    print("\nStatus: Tuning GaussianNB.")
    best_gini = 0
    best_vs = None

    for vs in var_smoothing_values:   # included to demonstrate that NB has no meaningful hyperparameters to tune
        model = GaussianNB(var_smoothing=vs)
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict_proba(X_valid_scaled)[:, 1]
        gini = 2 * roc_auc_score(y_valid, y_pred) - 1

        print(f"var_smoothing = {vs:.0e}: Gini={gini:.4f}")

        if gini > best_gini:
            best_gini = gini
            best_vs = vs
            best_model = model

    print(f"\nBest var_smoothing: {best_vs:.0e}")
    print(f"Best Gini: {best_gini:.4f}")
    print("Note: all values produced identical results (±0.0000)")

    return best_model, best_vs, best_gini

###### --------------------------------

# Calling functions
best_lr, lr_params, lr_gini = tune_logistic_regression(X_train, X_valid, y_train, y_valid)
best_knn, best_k, knn_gini = tune_knn(X_train, X_valid, y_train, y_valid)
best_nb, best_vs, nb_gini = tune_naive_bayes(X_train, X_valid, y_train, y_valid)

# Summary
print("- " * 40)
print("\nHyperparameters tuning summary")
print(f"LogisticRegression: C={lr_params['C']}, Gini={lr_gini:.4f}")
print(f"KNN:                k={best_k}, Gini={knn_gini:.4f}")
print(f"GaussianNB:         var_smoothing={best_vs:.0e}, Gini={nb_gini:.4f}")

print("\nHyperparameters impact analysis")
print(f"LogisticRegression (C): 0.0206 Gini impact")
print(f"KNN (k): 0.1252 Gini impact - most impactful")
print(f"GaussianNB (var_smoothing): 0.0000 Gini impact - no effect")


TASK 9. Hyperparameters tuning

Status: Tuning Logistic Regression.
C = 0.001: Gini=0.3488
C = 0.010: Gini=0.4742
C = 0.100: Gini=0.4638
C = 1.000: Gini=0.4636
C = 10.000: Gini=0.4626
C = 100.000: Gini=0.4624

Best params: {'C': 0.01}
Best Gini: 0.4742

Status: Tuning KNN.
k = 3: Gini=0.3061
k = 5: Gini=0.3228
k = 7: Gini=0.3403
k = 10: Gini=0.3563
k = 15: Gini=0.3806
k = 20: Gini=0.3955
k = 30: Gini=0.4085
k = 50: Gini=0.4313

Best k: 50
Best Gini: 0.4313

Status: Tuning GaussianNB.
var_smoothing = 1e-12: Gini=0.4491
var_smoothing = 1e-10: Gini=0.4491
var_smoothing = 1e-09: Gini=0.4491
var_smoothing = 1e-08: Gini=0.4491
var_smoothing = 1e-06: Gini=0.4491

Best var_smoothing: 1e-06
Best Gini: 0.4491
Note: all values produced identical results (±0.0000)
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

Hyperparameters tuning summary
LogisticRegression: C=0.01, Gini=0.4742
KNN:                k=50, Gini=0.4313
GaussianNB:         var_smoothing=1e-06, Gini=

In [14]:
# Best model
print(f"Best model: Logistic Regression with C = {lr_params['C']}")
print(f"Best Gini: {lr_gini:.4f}")
final_best_model = best_lr

Best model: Logistic Regression with C = 0.01
Best Gini: 0.4742


In [15]:
# TASK 10. Gini evaluation on all datasets (training, valid, test)
print("TASK 10. Gini evaluation on all datasets")

final_best_model = best_lr

def evaluate_all_datasets(final_best_model, X_train, X_valid, X_test,
                         y_train, y_valid, y_test):
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)
    X_test_scaled = scaler.transform(X_test)

    # Predictions
    y_train_pred = final_best_model.predict_proba(X_train_scaled)[:, 1]
    y_valid_pred = final_best_model.predict_proba(X_valid_scaled)[:, 1]
    y_test_pred = final_best_model.predict_proba(X_test_scaled)[:, 1]

    # Gini scores
    train_gini = 2 * roc_auc_score(y_train, y_train_pred) - 1
    valid_gini = 2 * roc_auc_score(y_valid, y_valid_pred) - 1
    test_gini = 2 * roc_auc_score(y_test, y_test_pred) - 1

    print(f"\nTrain Gini:      {train_gini:.4f}")
    print(f"Validation Gini: {valid_gini:.4f}")
    print(f"Test Gini:       {test_gini:.4f}")
    print(f"\nValid vs Test drop: {valid_gini - test_gini:.4f}")
    print(f"Train vs Valid gap: {train_gini - valid_gini:.4f}")

    # Analysis
    print("\nANALYSIS")
    if train_gini - valid_gini > 0.05:
        print("TRAIN vs. VALID: model shows signs of overfitting, train >> valid")
    else:
        print("TRAIN vs. VALID: model is not significantly overfitted")

    if abs(valid_gini - test_gini) < 0.02:
        print("VALID vs. TEST: good generalization from validation to test")
    else:
        print("VALID vs. TEST: some performance drop from validation to test")

    return {
        'train_gini': train_gini,
        'valid_gini': valid_gini,
        'test_gini': test_gini
    }

results_10 = evaluate_all_datasets(final_best_model, X_train, X_valid, X_test,
                         y_train, y_valid, y_test)

TASK 10. Gini evaluation on all datasets

Train Gini:      0.4981
Validation Gini: 0.4742
Test Gini:       0.3848

Valid vs Test drop: 0.0895
Train vs Valid gap: 0.0239

ANALYSIS
TRAIN vs. VALID: model is not significantly overfitted
VALID vs. TEST: some performance drop from validation to test


In [16]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# TASK 11. Recall, Precision, F1 score and AUC PR metrics calculation. Algorithms comparisom
print("TASK 11. Recall, Precision, F1 score and AUC PR metrics calculation. \nAlgorithms comparisom")

def calculate_precision_recall_f1(y_true, y_pred):
    # True Positives, False Positives, False Negatives
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))

    # Precision: TP / (TP + FP)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    # Recall: TP / (TP + FN)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    # F1: 2 * (Precision * Recall) / (Precision + Recall)
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1


def calculate_auc_pr(y_true, y_pred_proba):      ## calculating area under precision-recall curve
    # Sorting by predicted probability
    indices = np.argsort(y_pred_proba)[::-1]
    y_true_sorted = y_true[indices]

    # Calculating precision and recall at each threshold
    precisions = []
    recalls = []

    tp = 0
    fp = 0
    total_positives = np.sum(y_true)

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / total_positives if total_positives > 0 else 0

        precisions.append(precision)
        recalls.append(recall)

    # For AUC calculation
    precisions = np.array(precisions)
    recalls = np.array(recalls)

    # Sorting by recall
    sorted_indices = np.argsort(recalls)
    recalls = recalls[sorted_indices]
    precisions = precisions[sorted_indices]

    # AUC
    auc_pr = 0
    for i in range(1, len(recalls)):
        auc_pr += (recalls[i] - recalls[i-1]) * (precisions[i] + precisions[i-1]) / 2

    return auc_pr

# Extracting models from results dictionary (results dictionary - task 4)
models_to_compare = {
    'LogisticRegression': results['LogisticRegression']['model'],
    'GaussianNB': results['GaussianNB']['model'],
    'KNN': results['KNN']['model']
}

# Preparing test data
scaler = StandardScaler()

# Converting to numpy
if hasattr(X_test, 'values'):
    X_test_array = X_test.values
else:
    X_test_array = X_test

X_test_scaled = scaler.fit_transform(X_test_array)

# Storing results
comparison_results = []

threshold = 0.5  ## default threshold

for name, model in models_to_compare.items():
    print(f"\n{name}")

    # Predictions
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)

    # Calculating all metrics
    precision, recall, f1 = calculate_precision_recall_f1(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_pred_proba)
    gini = 2 * auc_roc - 1
    auc_pr = calculate_auc_pr(y_test, y_pred_proba)

    # Display
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc_roc:.4f}")
    print(f"Gini:      {gini:.4f}")
    print(f"AUC-PR:    {auc_pr:.4f}")

    comparison_results.append({
        'Model': name,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'AUC-ROC': auc_roc,
        'Gini': gini,
        'AUC-PR': auc_pr
    })

# Creating comparison DataFrame
comparison_df = pd.DataFrame(comparison_results)
print("\nSummary table:")
print(comparison_df.to_string(index=False))


TASK 11. Recall, Precision, F1 score and AUC PR metrics calculation. 
Algorithms comparisom

LogisticRegression
Precision: 0.8083
Recall:    0.2332
F1 Score:  0.3620
AUC-ROC:   0.7338
Gini:      0.4677
AUC-PR:    0.4254

GaussianNB
Precision: 0.1240
Recall:    1.0000
F1 Score:  0.2207
AUC-ROC:   0.5000
Gini:      0.0000
AUC-PR:    0.1337

KNN
Precision: 0.6608
Recall:    0.2594
F1 Score:  0.3725
AUC-ROC:   0.6717
Gini:      0.3433
AUC-PR:    0.3988

Summary table:
             Model  Precision   Recall       F1  AUC-ROC     Gini   AUC-PR
LogisticRegression   0.808266 0.233190 0.361954 0.733848 0.467695 0.425351
        GaussianNB   0.124050 1.000000 0.220719 0.500000 0.000000 0.133688
               KNN   0.660759 0.259357 0.372502 0.671657 0.343315 0.398818


TASK 12. Question: Which hard label metric do you prefer for detecting "lemon" cars?

Answer: Recall, as ovarall "costs" for buying a lemon are higher than for rejecting a good car. The costs mentioned include repairs, safety risks, and overpaid money.